# Progetto: prezzi delle case Iowa

Materiale originale **AI Lab**, non una lezione Kaggle. Codice in inglese; attività e spiegazioni in italiano.

Esegui le celle in ordine con **Maiusc + Invio**. Salva una copia prima di modificare il codice.


## Dati e obiettivo
Progetto originale AI Lab basato sui dati **Home Data for ML Course** usati dagli esercizi Kaggle. Non genera dati sostitutivi. Scarica il dataset con la guida del sito. L’obiettivo è confrontare una baseline con una random forest e salvare una previsione riproducibile. Richiede pandas e scikit-learn; il codice è predisposto ma non eseguito in questa sessione.


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
import joblib

project_root = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / "ai_lab/datasets.json").is_file()), None)
if project_root is None:
    raise FileNotFoundError("Open this notebook inside the extracted student bundle or repository.")
data_path = project_root / "ai_lab/input/home-data-for-ml-course/train.csv"
if not data_path.is_file():
    raise FileNotFoundError("Download the original Iowa dataset using the AI Lab data guide.")
home_data = pd.read_csv(data_path)
features = ["LotArea", "YearBuilt", "1stFlrSF", "2ndFlrSF", "FullBath", "BedroomAbvGr", "TotRmsAbvGrd"]
X, y = home_data[features], home_data["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.25, random_state=42)
print("Training / validation / test:", len(X_train), len(X_valid), len(X_test))


## Training e scelta
L’imputazione delle feature viene appresa dentro la pipeline solo sui dati di training. Confronta la baseline e la random forest sulla validazione. Scegli i parametri qui, senza leggere il test.


In [ ]:
baseline_prediction = float(y_train.mean())
print("Baseline validation MAE:", mean_absolute_error(y_valid, [baseline_prediction]*len(y_valid)))
model = make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=1))
model.fit(X_train, y_train)
print("Model validation MAE:", mean_absolute_error(y_valid, model.predict(X_valid)))


## Valutazione finale ed esportazione
Esegui solo dopo aver concluso le scelte. Il salvataggio include preprocessing e ordine delle feature. Carica con joblib soltanto artefatti fidati.


In [ ]:
print("Final test MAE:", mean_absolute_error(y_test, model.predict(X_test)))
artifact = {"model": model, "features": features, "target": "SalePrice"}
joblib.dump(artifact, "iowa-model.joblib")
restored = joblib.load("iowa-model.joblib")
sample = X_test.iloc[:1]
print("New prediction:", restored["model"].predict(sample))
assert abs(float(restored["model"].predict(sample)[0])-float(model.predict(sample)[0])) < 1e-8


## Attività
Confronta un albero e una random forest. Registra due configurazioni sulla validazione; esamina gli immobili con errori maggiori. Non dichiarare prestazioni su altre città senza nuovi dati.

## Interfaccia
Un modulo con i sette valori delle feature può richiamare la pipeline salvata tramite Gradio su un runtime Python esterno a GitHub Pages. Il modello non viene eseguito automaticamente dal sito statico.
